## Regresión Lineal para Predecir el LTV a 24 Meses

**Objetivo de aprendizaje:**
Descubrir variables que impactan el KPI de LTV a 24 meses mediante una regresión multivariable, e interpretar el modelo para apoyar la toma de decisiones.

### Contexto del caso: Tlacuachitos Express

Tlacuachitos Express es un servicio ficticio de entregas que desea predecir el valor que generará un cliente en los próximos 24 meses. El área de retención quiere utilizar este modelo para segmentar a los clientes y priorizar esfuerzos de fidelización.

### Exploración del KPI: ¿Qué es el LTV?

Discusión: ¿Qué es el Lifetime Value (LTV)? ¿Por qué es un KPI clave?

Definición del KPI: Suma de todas las compras realizadas por un cliente en sus primeros 24 meses.

### **1. Colección y comprensión de datos**

Se tiene 2 data-sets que contiene las siguientes variables:

* **Data-set customers**
    * *Customer ID:* Clave de identificación unica del cliente
    * *Age:* Edad del cliente.
    * *Income:* Ingreso mensual (MXN).
    * *Industry:* Isndutria en la que trabaja.
    * *Geographic Location:* Lugar donde vive el cliente.
    *  *Cohort:* Fecha en la que ingreso el cliente.

In [6]:
import pandas as pd

df_data = pd.read_csv('./resources/tlacuachitos_express_customers_data.csv')
df_data.head(3)

,CustomerID,Age,Income,Tenure,Education,Industry,Geographic Location,Cohort
0,1,56,52752.67735,3,Master,Technology,Europe,8/31/2023
1,2,69,55297.36435,6,Bachelor,Technology,South America,8/31/2021
2,3,46,57978.75338,3,Bachelor,Finance,Europe,5/31/2019


In [7]:
df_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1143 entries, 0 to 1142
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   CustomerID           1143 non-null   int64  
 1   Age                  1143 non-null   int64  
 2   Income               1143 non-null   float64
 3   Tenure               1143 non-null   int64  
 4   Education            1143 non-null   object 
 5   Industry             1143 non-null   object 
 6   Geographic Location  1143 non-null   object 
 7   Cohort               1143 non-null   object 
dtypes: float64(1), int64(3), object(4)
memory usage: 71.6+ KB


* **Data-set Transactions**
    * *Customer ID:* Clave de identificación unica del cliente
    * *Transaction Date:* Fecha en la que se realizo la compra.
    * *Transaction Amount:* Cantidad gastada en la compra (MXN).

In [8]:
df_transactions = pd.read_csv('./resources/tlacuachitos_express_transactions.csv')
df_transactions.head(3)

,CustomerID,TransactionDate,TransactionAmount
0,1,2023-08-31,524.891753
1,1,2024-10-31,794.653366
2,1,2023-12-31,223.096087


In [9]:
df_transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31552 entries, 0 to 31551
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         31552 non-null  int64  
 1   TransactionDate    31552 non-null  object 
 2   TransactionAmount  31552 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 739.6+ KB


### **2. Limpieza de datos**

* Conversión de 'TransactionDate' y 'Cohort' a formato de fecha.

In [10]:
df_transactions['TransactionDate'] = pd.to_datetime(df_transactions['TransactionDate'])
df_data['Cohort'] = pd.to_datetime(df_data['Cohort'])

* Fecha máxima de transacciónes, esta será considerada para el calculo de LTV.

In [11]:
snapshot_date = df_transactions['TransactionDate'].max()
snapshot_date


Timestamp('2025-03-31 00:00:00')

* Se revisa la fecha mínima de transacciónes.

In [12]:
df_transactions['TransactionDate'].min()

Timestamp('2018-01-31 00:00:00')

* Se calcula el 'customer_tenure' para saber la cantidad de meses que lleva el cliente en la empresa.
  * Diferencia de meses entre el 'cohort' (Fecha de ingreso) y snapshot_date (Fecha máxima | Marzo 2025).

In [13]:
df_data['customer_tenure'] = (snapshot_date.year - df_data['Cohort'].dt.year)*12 + (snapshot_date.month - df_data['Cohort'].dt.month)
df_data.head(3)

,CustomerID,Age,Income,Tenure,Education,Industry,Geographic Location,Cohort,customer_tenure
0,1,56,52752.67735,3,Master,Technology,Europe,2023-08-31,19
1,2,69,55297.36435,6,Bachelor,Technology,South America,2021-08-31,43
2,3,46,57978.75338,3,Bachelor,Finance,Europe,2019-05-31,70


* Se cruza la informacion de los clientes a la base de *df_transactions*.

In [14]:
df_master = df_transactions.merge(df_data, on='CustomerID')
df_master.head(3)

,CustomerID,TransactionDate,TransactionAmount,Age,Income,Tenure,Education,Industry,Geographic Location,Cohort,customer_tenure
0,1,2023-08-31,524.891753,56,52752.67735,3,Master,Technology,Europe,2023-08-31,19
1,1,2024-10-31,794.653366,56,52752.67735,3,Master,Technology,Europe,2023-08-31,19
2,1,2023-12-31,223.096087,56,52752.67735,3,Master,Technology,Europe,2023-08-31,19


* Se calcula la diferencia en meses entre la transacciones de los clientes y el cohort 'customer_tenure_on_transaction'.
  * Tenure del cliente cuando realizo la orden.

In [15]:
df_master['customer_tenure_on_transaction'] = (
    df_master['TransactionDate'].dt.year - df_master['Cohort'].dt.year)*12 + (
    df_master['TransactionDate'].dt.month - df_master['Cohort'].dt.month
)
df_master.head()

,CustomerID,TransactionDate,TransactionAmount,Age,Income,Tenure,Education,Industry,Geographic Location,Cohort,customer_tenure,customer_tenure_on_transaction
0,1,2023-08-31,524.891753,56,52752.67735,3,Master,Technology,Europe,2023-08-31,19,0
1,1,2024-10-31,794.653366,56,52752.67735,3,Master,Technology,Europe,2023-08-31,19,14
2,1,2023-12-31,223.096087,56,52752.67735,3,Master,Technology,Europe,2023-08-31,19,4
3,1,2023-12-31,458.998091,56,52752.67735,3,Master,Technology,Europe,2023-08-31,19,4
4,1,2024-03-31,127.133098,56,52752.67735,3,Master,Technology,Europe,2023-08-31,19,7


In [16]:
df_master[['Cohort', 'TransactionDate', 'customer_tenure_on_transaction']].head()

,Cohort,TransactionDate,customer_tenure_on_transaction
0,2023-08-31,2023-08-31,0
1,2023-08-31,2024-10-31,14
2,2023-08-31,2023-12-31,4
3,2023-08-31,2023-12-31,4
4,2023-08-31,2024-03-31,7


### **3. Cálculo del KPI objetivo: LTV a 24 meses**

In [17]:
df_master.columns

Index(['CustomerID', 'TransactionDate', 'TransactionAmount', 'Age', 'Income',
       'Tenure', 'Education', 'Industry', 'Geographic Location', 'Cohort',
       'customer_tenure', 'customer_tenure_on_transaction'],
      dtype='object')

* **LTV 24 meses:** Compras en los primeros 24 meses de los clientes.
    * Clientes con al menos 24 meses en la empresa (['customer_tenure'] > 24)
    * Ordenes dentro de los primero 24 meses del cliente (['customer_tenure_on_transaction'] <= 24)

In [18]:
cltv_24_months = df_master[
    (df_master['customer_tenure'] > 24) &
    (df_master['customer_tenure_on_transaction'] <= 24)
].groupby('CustomerID')['TransactionAmount'].sum().reset_index()

cltv_24_months.rename(columns={'TransactionAmount': 'LTV'}, inplace=True)
cltv_24_months.head()

,CustomerID,LTV
0,2,11276.581901
1,3,5084.632444
2,4,3037.917187
3,5,11677.948404
4,6,4556.353725


### **4. Ingeniería de características**

Se identifican las variables numericas y las categoricas.

In [19]:
categorical_features = ['Education', 'Industry', 'Geographic Location']
numerical_features = ['Age', 'Income']

**Variables dummies:** variables artificiales que se usan para representar información categórica en modelos matemáticos.

* Se crea una columna por cada categoría, menos una (se omite una como referencia o base).
* Esto evita la colinealidad perfecta en modelos lineales (por ejemplo, en regresión).
* Se usa típicamente en estadística y modelos lineales clásicos.

In [20]:
df_master['Education'].unique()

array(['Master', 'Bachelor', 'High School', 'PhD'], dtype=object)

In [21]:
df_master['Industry'].unique()

array(['Technology', 'Finance', 'Education', 'Entertainment',
       'Healthcare'], dtype=object)

In [22]:
df_master['Geographic Location'].unique()

array(['Europe', 'South America', 'Asia', 'North America', 'Australia'],
      dtype=object)

In [23]:

df_encoded = pd.get_dummies(df_data, columns=categorical_features, drop_first=True)
df_encoded.head()

,CustomerID,Age,Income,Tenure,Cohort,customer_tenure,Education_High School,Education_Master,Education_PhD,Industry_Entertainment,Industry_Finance,Industry_Healthcare,Industry_Technology,Geographic Location_Australia,Geographic Location_Europe,Geographic Location_North America,Geographic Location_South America
0,1,56,52752.67735,3,2023-08-31,19,False,True,False,False,False,False,True,False,True,False,False
1,2,69,55297.36435,6,2021-08-31,43,False,False,False,False,False,False,True,False,False,False,True
2,3,46,57978.75338,3,2019-05-31,70,False,False,False,False,True,False,False,False,True,False,False
3,4,32,60445.26690,3,2021-02-28,49,True,False,False,False,False,False,False,False,False,False,True
4,5,60,57741.87093,5,2018-10-31,77,False,False,False,True,False,False,False,False,False,False,False


In [24]:
df_encoded.columns

Index(['CustomerID', 'Age', 'Income', 'Tenure', 'Cohort', 'customer_tenure',
       'Education_High School', 'Education_Master', 'Education_PhD',
       'Industry_Entertainment', 'Industry_Finance', 'Industry_Healthcare',
       'Industry_Technology', 'Geographic Location_Australia',
       'Geographic Location_Europe', 'Geographic Location_North America',
       'Geographic Location_South America'],
      dtype='object')

* Una vez que ya se tiene la base con las variables dummies y categóricas, se cruza el LTV a 24 meses.

In [25]:
cltv_24_months.head()

,CustomerID,LTV
0,2,11276.581901
1,3,5084.632444
2,4,3037.917187
3,5,11677.948404
4,6,4556.353725


In [26]:

df_to_model = cltv_24_months.merge(df_encoded, on='CustomerID')
df_to_model.head()

,CustomerID,LTV,Age,Income,Tenure,Cohort,customer_tenure,Education_High School,Education_Master,Education_PhD,Industry_Entertainment,Industry_Finance,Industry_Healthcare,Industry_Technology,Geographic Location_Australia,Geographic Location_Europe,Geographic Location_North America,Geographic Location_South America
0,2,11276.581901,69,55297.36435,6,2021-08-31,43,False,False,False,False,False,False,True,False,False,False,True
1,3,5084.632444,46,57978.75338,3,2019-05-31,70,False,False,False,False,True,False,False,False,True,False,False
2,4,3037.917187,32,60445.26690,3,2021-02-28,49,True,False,False,False,False,False,False,False,False,False,True
3,5,11677.948404,60,57741.87093,5,2018-10-31,77,False,False,False,True,False,False,False,False,False,False,False
4,6,4556.353725,25,57132.40462,3,2022-06-30,33,False,True,False,False,False,True,False,False,False,True,False


* Guardar el nombre de las variables dummy en una variable.

In [27]:
dummy_features = (
    df_to_model.columns[df_to_model.columns.str.startswith(tuple(categorical_features))].values).tolist()

In [28]:
dummy_features

['Education_High School',
 'Education_Master',
 'Education_PhD',
 'Industry_Entertainment',
 'Industry_Finance',
 'Industry_Healthcare',
 'Industry_Technology',
 'Geographic Location_Australia',
 'Geographic Location_Europe',
 'Geographic Location_North America',
 'Geographic Location_South America']

### **5. Regresión lineal multivariable**

La **regresión lineal múltiple** es una técnica estadística que permite analizar la relación entre una **variable dependiente** ($y$) y **dos o más variables independientes** ( $x_{1}$, $x_{2}$, $\dots$, $x_{p}$ ), bajo el supuesto de que dicha relación puede modelarse de forma **lineal**.

<center>
    <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Linear_regression.svg/350px-Linear_regression.svg.png"  width="300" />
</center> 

📌 **Modelo**

$$y_{i} = \beta_{0} + \beta_{1} x_{i1} + \beta_2 x_{i2} + \dots + \beta_{p} x_{ip} + \varepsilon_i$$

Donde:

- $y_{i}$: valor de la variable dependiente en la observación \( i \)
- $x_{ij}$: valor de la j-ésima variable independiente en la observación \( i \)
- $\beta_{0}$: intercepto del modelo
- $\beta_{j}$: coeficientes del modelo (efecto de cada variable \( x_j \))
- $\varepsilon_i$: error aleatorio (lo que el modelo no puede explicar)

🎯 **Objetivo**

Estimar los parámetros $\beta_{0}$, $\beta_{1}$, $\dots$, $\beta_{p}$ de manera que la ecuación:

$$\hat{y}_i = \hat{\beta}_0 + \hat{\beta}_1 x_{i1} + \hat{\beta}_2 x_{i2} + \dots + \hat{\beta}_p x_{ip}$$

**minimice el error total** entre los valores reales $y_{i}$ y los valores predichos $\hat{y}_i$, usando el **método de mínimos cuadrados ordinarios (OLS)**:

$$\min \sum_{i=1}^n (y_i - \hat{y}_i)^2$$

In [29]:
#pip install statsmodels

In [30]:
import statsmodels.api as sm

X = df_to_model[dummy_features + numerical_features]
y = ['LTV']

#Agrega una variable como constante para el intercepto
X = sm.add_constant(X)

model = sm.OLS(df_to_model[y], X.astype(float))
results = model.fit()

results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                    LTV   R-squared:                       0.615
Model:                            OLS   Adj. R-squared:                  0.610
Method:                 Least Squares   F-statistic:                     119.2
Date:                Tue, 08 Apr 2025   Prob (F-statistic):          1.44e-190
Time:                        20:42:20   Log-Likelihood:                -8875.8
No. Observations:                 984   AIC:                         1.778e+04
Df Residuals:                     970   BIC:                         1.785e+04
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
=====================================================================================================
                                        coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
const                             -1621.4359    352.703     -4.597      0.000   -2313.585    -929.287
Education_High School             -1539.1595    156.971     -9.805      0.000   -1847.201   -1231.118
Education_Master                   1394.9743    176.017      7.925      0.000    1049.555    1740.393
Education_PhD                      2125.8119    229.469      9.264      0.000    1675.498    2576.125
Industry_Entertainment             2689.3240    203.256     13.231      0.000    2290.452    3088.196
Industry_Finance                   1094.0132    202.207      5.410      0.000     697.200    1490.826
Industry_Healthcare                 409.9379    196.538      2.086      0.037      24.248     795.627
Industry_Technology                1956.0130    200.716      9.745      0.000    1562.124    2349.902
Geographic Location_Australia       -66.9734    204.681     -0.327      0.744    -468.643     334.696
Geographic Location_Europe          191.9400    207.613      0.925      0.355    -215.482     599.362
Geographic Location_North America   731.0676    212.564      3.439      0.001     313.930    1148.205
Geographic Location_South America   171.9923    207.475      0.829      0.407    -235.160     579.145
Age                                 128.9788      4.257     30.298      0.000     120.625     137.333
Income                                0.0323      0.004      7.744      0.000       0.024       0.040
==============================================================================
Omnibus:                       39.813   Durbin-Watson:                   2.012
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               55.753
Skew:                           0.377   Prob(JB):                     7.82e-13
Kurtosis:                       3.889   Cond. No.                     3.45e+05
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 3.45e+05. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

----------
* **$R^{2}$ | Coeficiente de determinación**

Métrica que indica proporción de la variabilidad de la variable dependiente $y$ es explicada por el modelo (variables independientes $x$).

- $R^{2} = 1$: el modelo explica el 100% de la variabilidad de $y$.  
- $R^{2} = 0$: el modelo no explica nada; equivale a predecir con el promedio de $y$.
- $0 < R^{2} < 1$: el modelo explica parcialmente la variabilidad de $y$.

* **$R^{2} Ajustado$ | Coeficiente de determinación Ajustado**

La interpretación de esta métrica es igual que el $R^{2}$, sin embargo el $R^{2} Ajustado$ corrige este valor penalizando la inclusión de variables irrelevantes.

In [31]:
print(f"R² = El modelo explica el {results.rsquared * 100:.2f}% de la variabilidad del LTV a 24 meses.")

R² = El modelo explica el 61.49% de la variabilidad del LTV a 24 meses.


In [32]:
print(f"R² ajustado = El modelo explica el {results.rsquared_adj * 100:.2f}% de la variabilidad del LTV a 24 meses, considerando la penalización por variables irrelevantes.")

R² ajustado = El modelo explica el 60.98% de la variabilidad del LTV a 24 meses, considerando la penalización por variables irrelevantes.


----------
* **F-statistic**
Prueba F evalúa la **significancia global del modelo de regresión**, es decir, si al menos una de las variables predictoras $x_{1}, x_{2}, \dots, x_{p}$ tiene una relación significativa con la variable dependiente $y$.
  * **Hipótesis nula $H_{0}$:**  
  Ninguna de las variables predictoras $x_{1}, x_{2}, \dots, x_{p}$ explica significativamente la variabilidad en $y$.  
  $H_{0}: \beta_{1} = \beta_{2} = \dots = \beta_{p} = 0$

  * **Hipótesis alternativa $H_{1}$:**  
  Al menos una de las variables predictoras tiene una relación significativa con $y$.  
  $H_1: \text{Al menos uno de los } \beta_{j} \neq 0$

* **Prob(F-statistic)** p-value asociado a la prueba F.
  * *p-value < 0.05:* se rechaza $H_{0}$
  * *p-value >= 0.05:* se acepta $H_{0}$



In [33]:
if results.f_pvalue < 0.05:
    print("Con un p-value menor a 0.05 se rechaza H0, por lo tanto al menos una de las variables predictoras tiene una relación significativa con $y$.")
else:
    print("Con un p-value mayor a 0.05 se acepta H0, por lo tanto ninguna de la vairables predictoras explica significativamente la vairabilidad en $y$.")

Con un p-value menor a 0.05 se rechaza H0, por lo tanto al menos una de las variables predictoras tiene una relación significativa con $y$.


----------
* **Coeficientes**
Cada variable predictora esta acompañada de un coeficiente y un p-value.
  * **Hipótesis nula $H_{0}$:**  
  La variable predictora **no tiene** una relación significativa con la variable dependiente $y$. Esto significa que el coeficiente asociado es igual a cero.  
  $H_{0}: \beta_{j} = 0$

  * **Hipótesis alternativa $H_{1}$:**  
  La variable predictora **sí tiene** una relación significativa con $y$. Esto significa que el coeficiente asociado es diferente de cero.  
  $H_1: \beta_{j} \neq 0$

* **P>|t|:** p-value asociado a la prueba.
  * *p-value < 0.05:* se rechaza $H_{0}$
  * *p-value >= 0.05:* se acepta $H_{0}$

In [34]:
#Vairables predictivas que tienen una relación significativa con el LTV a 24 meses
results.pvalues[results.pvalues<0.05]

const                                 4.848407e-06
Education_High School                 1.046434e-21
Education_Master                      6.228904e-15
Education_PhD                         1.233815e-19
Industry_Entertainment                7.335113e-37
Industry_Finance                      7.925146e-08
Industry_Healthcare                   3.725845e-02
Industry_Technology                   1.798581e-21
Geographic Location_North America     6.080689e-04
Age                                  1.943837e-142
Income                                2.418621e-14
dtype: float64

In [42]:
results.pvalues[results.pvalues>=0.05]

Geographic Location_Australia        0.743581
Geographic Location_Europe           0.355451
Geographic Location_South America    0.407322
dtype: float64

-----------------------------
* **Interpretación de los coeficientes**

Dado que estamos trabajando con un modelo lineal, los coeficientes asociados a cada variable predictora pueden interpretarse para entender su relación con la variable dependiente (o "objetivo"). Esta interpretación depende del tipo de variable (numérica o categórica).

- **Variables numéricas:**
  - **Coeficiente positivo:** Por cada unidad que incrementa la variable predictora, la variable objetivo **aumenta** en $\beta$.
  - **Coeficiente negativo:** Por cada unidad que incrementa la variable predictora, la variable objetivo **disminuye** en $\beta$.

**Manteniendo el resto de las variables constantes.**


In [35]:
#Solo aquellos coeficientes significativos
results.params[results.pvalues[results.pvalues<0.05].index]

const                               -1621.435943
Education_High School               -1539.159508
Education_Master                     1394.974258
Education_PhD                        2125.811934
Industry_Entertainment               2689.323988
Industry_Finance                     1094.013197
Industry_Healthcare                   409.937865
Industry_Technology                  1956.012992
Geographic Location_North America     731.067626
Age                                   128.978776
Income                                  0.032310
dtype: float64

In [36]:
print(f"Por cada año que incremente la edad de los clientes, el LTV aumenta {results.params['Age']:.2f} MXN.")

Por cada año que incremente la edad de los clientes, el LTV aumenta 128.98 MXN.


In [37]:
print(f"Por cada peso (MXN) que incremente la ingreso mesual de los clientes, el LTV aumenta {results.params['Income']:.2f} MXN.")

Por cada peso (MXN) que incremente la ingreso mesual de los clientes, el LTV aumenta 0.03 MXN.


- **Variables Categóricas:**
  - **Coeficiente positivo:** significa que la categoría tiene un valor más alto que el valor de referencia.
  - **Coeficiente negativo:** significa que la categoría tiene un valor más bajo que el valor de referencia.

**Manteniendo el resto de las variables constantes.**

In [38]:
df_data['Education'].unique()

array(['Master', 'Bachelor', 'High School', 'PhD'], dtype=object)

Para la variable 'Education' el valor en referencia es 'Bachelor'.

- **Valor de referencia (Bachelor)**: En este caso, el valor de referencia es **'Bachelor'**. Esto significa que los coeficientes de las demás categorías se interpretan en comparación con **'Bachelor'**.

- **Education_High School (-1,539.15 MXN)**: Los clientes con **High School** tienen un LTV **a 24 meses** que es **1,539.15 MXN menos** que los clientes con **Bachelor**, manteniendo constantes las otras variables del modelo.

- **Education_Master (-1,394.97 MXN)**: Los clientes con **Master** tienen un LTV **a 24 meses** que es **1,394.97 MXN menos** que los clientes con **Bachelor**.

- **Education_PhD (2,125.8 MXN)**: Los clientes con **PhD** tienen un LTV **a 24 meses** que es **2,125.8 MXN más** que los clientes con **Bachelor**.

**Manteniendo el resto de las variables constantes.**

In [39]:
results.params[results.pvalues[results.pvalues<0.05].index]

const                               -1621.435943
Education_High School               -1539.159508
Education_Master                     1394.974258
Education_PhD                        2125.811934
Industry_Entertainment               2689.323988
Industry_Finance                     1094.013197
Industry_Healthcare                   409.937865
Industry_Technology                  1956.012992
Geographic Location_North America     731.067626
Age                                   128.978776
Income                                  0.032310
dtype: float64

### **6. Interpretación del modelo**
- ¿Qué significa que las variables sean sean estadísticamente significativas?.
- ¿Qué variables son estadísticamente significativas?.
- ¿Qué coeficientes son positivos o negativos?.
- Interpreta el resto de los coeficientes estadisticamente significativos.

### **7. Predicción de un nuevo cliente**


In [40]:
new_customer_data = {
    'const': [1],
    'Education_High School': [0],
    'Education_Master': [0],
    'Education_PhD': [0],
    'Industry_Entertainment': [0],
    'Industry_Finance': [0],
    'Industry_Healthcare': [1],
    'Industry_Technology': [0],
    'Geographic Location_Australia': [0],
    'Geographic Location_Europe': [0],
    'Geographic Location_North America': [0],
    'Geographic Location_South America': [1],
    'Age': [18],
    'Income': [3500],
}

new_customer_df = pd.DataFrame(new_customer_data)

predicted_ltv = results.predict(new_customer_df)

print(f"Predicted LTV for the new customer: ${predicted_ltv[0]:,.2f}")


Predicted LTV for the new customer: $1,395.20


### **Laboratorio 7**

**Con base en el análisis realizado sobre la predicción de LTV:**

1.- Responde las preguntas del apartado 6 e interpreta los coeficientes restantes.

2.- Sube el archivo con todas las secciones ejecutadas.